# 02 — Goal 1: occurrence and acquisition variability

Audits eligibility, metric support, sparse-event prevalence, distributions, and exploratory acquisition/cohort contrasts.

Every displayed denominator and paper-facing visual is also saved under separate `figures/` and `tables/` folders within `outputs/visualization/`. Empty or under-supported analyses remain visible as audit rows; they are never silently removed. The final cell explains whether the next stage is allowed.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
VIZ_ROOT = OUTPUT / "visualization"
sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def read_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing required stage table: {parquet} or {csv}")

def read_optional_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    if stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists():
        return read_stage(relative_without_suffix)
    print("OPTIONAL TABLE NOT AVAILABLE:", relative_without_suffix)
    return pd.DataFrame()

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def save_table(frame, folder, name):
    target = VIZ_ROOT / folder / "tables"
    target.mkdir(parents=True, exist_ok=True)
    path = target / f"{name}.csv"
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, folder, name):
    target = VIZ_ROOT / folder / "figures"
    target.mkdir(parents=True, exist_ok=True)
    png = target / f"{name}.png"
    svg = target / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)
print("Visualization outputs:", VIZ_ROOT)


In [ ]:
RUN_GOAL_1 = False
if RUN_GOAL_1:
    run_cli("extract", "--profile", "primary")
    run_cli("assemble")
    run_cli("describe")


In [ ]:
from paper1_qc.registry import metric_registry_frame

data = read_stage("03_dataset_assembly/paper1_analysis_dataset")
flow = read_stage("03_dataset_assembly/eligibility_flow_counts")
descriptive = read_stage("04_analysis/descriptive/metric_descriptive_statistics")
contrasts = read_stage("04_analysis/descriptive/exploratory_participant_level_diagnosis_contrasts")
registry = metric_registry_frame()

display(flow)
save_table(flow, "02_goal1", "eligibility_flow_counts")

eligible = data.loc[data["primary_measurement_eligible"].fillna(False)].copy()
cohort = (
    eligible.groupby("diagnosis_analysis", dropna=False)
    .agg(recordings=("logical_recording_id", "nunique"), participants=("SubjectID", "nunique"))
    .reset_index()
)
cohort["recordings_per_participant"] = cohort["recordings"] / cohort["participants"]
save_table(cohort, "02_goal1", "cohort_imbalance")
display(cohort)


In [ ]:
# Metric support is an outcome, not a nuisance to hide.
support = descriptive.merge(
    registry[["feature", "family", "unit", "role", "worse", "minimum_support"]],
    on=["feature", "family", "role"],
    how="left",
    validate="one_to_one",
)
support["supported_fraction"] = support["recordings_nonmissing"] / len(eligible)
support = support.sort_values(["family", "supported_fraction", "feature"])
save_table(support, "02_goal1", "metric_support_and_descriptives")
display(support)

fig, ax = plt.subplots(figsize=(11, max(6, 0.27 * len(support))))
sns.scatterplot(
    data=support,
    x="supported_fraction",
    y="feature",
    hue="family",
    style="role",
    s=70,
    ax=ax,
)
ax.axvline(.8, color="0.4", linestyle="--", linewidth=1)
ax.set(title="Metric-specific support denominators", xlabel="Fraction of eligible recordings", ylabel="")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
save_figure(fig, "02_goal1", "metric_support_fraction")
plt.show()


In [ ]:
# Family-faceted raw-unit distributions. No cross-unit composite is plotted.
metric_columns = [feature for feature in registry["feature"] if feature in eligible]
long = eligible[["file_name", "SubjectID", "diagnosis_analysis", *metric_columns]].melt(
    id_vars=["file_name", "SubjectID", "diagnosis_analysis"],
    var_name="feature",
    value_name="value",
)
long = long.merge(registry[["feature", "family", "unit", "role"]], on="feature", how="left")

for family, family_frame in long.groupby("family", sort=True):
    selected = family_frame["feature"].drop_duplicates().tolist()
    ncols = 2
    nrows = int(np.ceil(len(selected) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.3 * nrows), squeeze=False)
    for ax, feature in zip(axes.flat, selected):
        plot_data = family_frame.loc[family_frame["feature"].eq(feature)]
        sns.histplot(
            data=plot_data,
            x="value",
            hue="diagnosis_analysis",
            element="step",
            stat="density",
            common_norm=False,
            bins=25,
            ax=ax,
        )
        unit = registry.set_index("feature").loc[feature, "unit"]
        ax.set(title=feature, xlabel=unit, ylabel="Density")
    for ax in axes.flat[len(selected):]:
        ax.axis("off")
    fig.suptitle(f"Raw metric distributions — {family}", y=1.01)
    fig.tight_layout()
    save_figure(fig, "02_goal1", f"raw_distributions__{family}")
    plt.show()


In [ ]:
# Sparse-event table: absence is zero only when the metric extraction status had support.
sparse = support.loc[support["zero_fraction_nonmissing"].ge(.5)].copy()
sparse["recording_event_prevalence"] = 1 - sparse["zero_fraction_nonmissing"]
sparse["interpretation_gate"] = np.where(
    sparse["supported_fraction"].ge(.8),
    "report prevalence and positive magnitude",
    "support-limited; report missingness first",
)
save_table(sparse, "02_goal1", "sparse_event_prevalence")
display(sparse[[
    "feature", "family", "recordings_nonmissing", "supported_fraction",
    "recording_event_prevalence", "interpretation_gate"
]])

fig, ax = plt.subplots(figsize=(10, max(4, .35 * len(sparse))))
sns.barplot(data=sparse, x="recording_event_prevalence", y="feature", hue="family", ax=ax)
ax.set(title="Observed event prevalence among supported recordings", xlabel="Prevalence", ylabel="")
save_figure(fig, "02_goal1", "sparse_event_prevalence")
plt.show()


In [ ]:
# Exploratory participant-level ALS-control effects; counts and CI status stay attached.
estimable = contrasts.loc[contrasts["status"].eq("ok")].copy()
save_table(contrasts, "02_goal1", "participant_level_diagnosis_contrasts")
fig, ax = plt.subplots(figsize=(11, max(6, .3 * len(estimable))))
ax.errorbar(
    estimable["cliffs_delta_a_vs_b"],
    np.arange(len(estimable)),
    xerr=np.vstack([
        estimable["cliffs_delta_a_vs_b"] - estimable["cliffs_delta_ci_low"],
        estimable["cliffs_delta_ci_high"] - estimable["cliffs_delta_a_vs_b"],
    ]),
    fmt="o",
    color="0.2",
    ecolor="0.55",
    capsize=2,
)
ax.axvline(0, color="0.3", linestyle="--")
ax.set_yticks(np.arange(len(estimable)), estimable["feature"])
ax.set(title="Exploratory participant-level ALS vs control contrasts", xlabel="Cliff's delta", ylabel="")
save_figure(fig, "02_goal1", "diagnosis_cliffs_delta_forest")
plt.show()


In [ ]:
feature_errors = read_optional_stage("02_features/feature_extraction_errors")
goal1_reasons = []
if eligible.empty:
    goal1_reasons.append("No recordings satisfy primary measurement eligibility.")
if support.empty:
    goal1_reasons.append("Metric support table is empty.")
if not feature_errors.empty:
    goal1_reasons.append(
        f"{len(feature_errors)} feature-extraction errors require correction or documented exclusion."
    )
goal1_ready = stage_gate(
    "Goal 1 occurrence/acquisition variability",
    not goal1_reasons,
    goal1_reasons,
    "Open 03_goal2_participant_persistence.ipynb only after this gate passes.",
)